# S23DR 2026 — RoofWireframeNet Training

**Runtime**: GPU (Runtime → Change runtime type → T4 GPU)  
**Expected time**: ~3.6 min/epoch on T4, ~1.2 min/epoch on A100

**Resume workflow** — works across session resets automatically:
1. Run cells 1–5 (setup + smoke test)
2. Run cell 6 (training) — auto-restores from Drive if a previous checkpoint exists
3. Run cell 7 (save to Drive) after each session — keeps your progress safe

Resume priority: `last.pt` → `best.pt` → start fresh

In [ ]:
# ── 1. Check GPU ─────────────────────────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── 2. Install dependencies ──────────────────────────────────────────────────
!pip install -q datasets huggingface_hub scipy

In [ ]:
# ── 3. Clone / update repo ───────────────────────────────────────────────────
import os
if not os.path.exists('3d_building_construction'):
    !git clone https://github.com/12turtleships/3d_building_construction.git
%cd 3d_building_construction
!git pull
!git log --oneline -3

In [ ]:
# ── 4. Verify imports ────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '.')
from s23dr.model import RoofWireframeNet, WireframeLoss, N_EDGE_CLASSES
from s23dr.data  import S23DRDataset, collate_fn

model = RoofWireframeNet(n_queries=64)
total = sum(p.numel() for p in model.parameters())
print(f'Model: {total:,} parameters')
print('OK')

In [ ]:
# ── 5. Smoke test ────────────────────────────────────────────────────────────
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using:', device)

model = RoofWireframeNet(n_queries=64).to(device)
loss_fn = WireframeLoss()

B, N = 4, 1024
xyz  = torch.randn(B, N, 3).to(device)
vf   = torch.rand(B, N).to(device)
nv   = torch.randint(0, 8, (B, N)).float().to(device)
msk  = torch.ones(B, N).to(device)
cid  = torch.randint(0, 10, (B, N)).to(device)
gv   = [torch.randn(31, 3) for _ in range(B)]
ge   = [torch.randint(0, 31, (32, 2)) for _ in range(B)]
gc   = [torch.randint(0, 10, (32,)) for _ in range(B)]

out    = model(xyz, vf, nv, msk, cid)
losses = loss_fn(out['pred_pos'], out['pred_conf'], out['edge_logits'], gv, ge, gc)
print('pred_pos:', tuple(out['pred_pos'].shape))
print('loss:    ', losses['loss'].item())
print('Smoke test passed!')

In [ ]:
# ── 6. Full training (auto-resume) ───────────────────────────────────────────
#
# On every new session this cell:
#   1. Mounts Google Drive and copies s23dr_last.pt → outputs/checkpoints/last.pt
#      (skipped silently if Drive has no checkpoint yet)
#   2. Resumes from last.pt if present, else falls back to best.pt, else starts fresh
#   3. Saves last.pt + best.pt locally after every epoch
#   4. Also syncs last.pt to Drive every DRIVE_SYNC_EVERY epochs
#
# T4  (free Colab):  BATCH_SIZE=8,  EPOCHS=100 → ~6 hrs
# A100 (Colab Pro+): BATCH_SIZE=16, EPOCHS=100 → ~2 hrs

import shutil, time
from pathlib import Path
from torch.utils.data import DataLoader
import torch
import torch.optim as optim

EPOCHS           = 100
BATCH_SIZE       = 8
LR               = 1e-3
N_QUERIES        = 64
DRIVE_SYNC_EVERY = 5          # save to Drive every N epochs
CKPT_DIR         = Path('outputs/checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LAST_CKPT        = CKPT_DIR / 'last.pt'
BEST_CKPT        = CKPT_DIR / 'best.pt'
DRIVE_LAST       = '/content/drive/MyDrive/s23dr_last.pt'
DRIVE_BEST       = '/content/drive/MyDrive/s23dr_best.pt'

# ── helpers ──
def _torch_load(path, device='cpu'):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)

def _save(path, epoch, val_loss):
    torch.save({'epoch': epoch, 'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'val_loss': val_loss}, path)

# ── 1. restore last.pt from Drive if available ──
try:
    from google.colab import drive as _drive
    _drive.mount('/content/drive', force_remount=False)
    drive_available = True
except Exception:
    drive_available = False

if drive_available and not LAST_CKPT.exists():
    for src, dst in [(DRIVE_LAST, LAST_CKPT), (DRIVE_BEST, BEST_CKPT)]:
        if Path(src).exists():
            shutil.copy(src, dst)
            print(f'Restored {Path(src).name} from Drive')

# ── 2. build model / optimizer / scheduler ──
model     = RoofWireframeNet(n_queries=N_QUERIES).to(device)
loss_fn   = WireframeLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

start_epoch = 1
best_val    = float('inf')

# ── 3. resume — last.pt first, then best.pt ──
for ckpt_path, label in [(LAST_CKPT, 'last.pt'), (BEST_CKPT, 'best.pt')]:
    if ckpt_path.exists():
        ckpt = _torch_load(ckpt_path, device)
        model.load_state_dict(ckpt['model'])
        if 'optimizer' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer'])
        if 'scheduler' in ckpt:
            scheduler.load_state_dict(ckpt['scheduler'])
        else:
            # old-format checkpoint — fast-forward scheduler to match saved epoch
            for _ in range(ckpt['epoch']):
                scheduler.step()
        start_epoch = ckpt['epoch'] + 1
        best_val    = ckpt.get('val_loss', float('inf'))
        print(f'Resumed from {label}  epoch={ckpt["epoch"]}  '
              f'val_loss={best_val:.4f}  → continuing from epoch {start_epoch}')
        # Keep best_val in sync with best.pt even when resuming from last.pt
        if label == 'last.pt' and BEST_CKPT.exists():
            best_val = _torch_load(BEST_CKPT).get('val_loss', best_val)
        break
else:
    print(f'Starting fresh — {sum(p.numel() for p in model.parameters()):,} params on {device}')

# ── 4. load dataset ──
print('Loading dataset…')
train_ds = S23DRDataset(split='train',      n_points=1024)
val_ds   = S23DRDataset(split='validation', n_points=1024)
print(f'Train: {len(train_ds)}  Val: {len(val_ds)}')
print(f'{EPOCHS - start_epoch + 1} epochs remaining × '
      f'{len(train_ds) // BATCH_SIZE} steps/epoch')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, collate_fn=collate_fn, drop_last=True,
                          pin_memory=(device=='cuda'))
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, collate_fn=collate_fn,
                          pin_memory=(device=='cuda'))

LOG_EVERY = 200

# ── 5. training loop ──
for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    t0 = time.time()
    for step, batch in enumerate(train_loader):
        xyz  = batch['xyz'].to(device)
        out  = model(xyz, batch['vote_frac'].to(device),
                     batch['n_views'].to(device), batch['mask'].to(device),
                     batch['class_id'].to(device))
        loss = loss_fn(out['pred_pos'], out['pred_conf'], out['edge_logits'],
                       batch['gt_verts'], batch['gt_edges'], batch['gt_classes'])['loss']
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
        if (step + 1) % LOG_EVERY == 0:
            print(f'  ep{epoch} step{step+1}/{len(train_loader)}  '
                  f'loss={loss.item():.4f}  ({time.time()-t0:.0f}s)')

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            out = model(batch['xyz'].to(device), batch['vote_frac'].to(device),
                        batch['n_views'].to(device), batch['mask'].to(device),
                        batch['class_id'].to(device))
            val_loss += loss_fn(out['pred_pos'], out['pred_conf'], out['edge_logits'],
                                batch['gt_verts'], batch['gt_edges'],
                                batch['gt_classes'])['loss'].item()
    val_loss /= len(val_loader)
    train_loss = epoch_loss / len(train_loader)
    scheduler.step()
    print(f'Epoch {epoch:3d}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}  '
          f'lr={scheduler.get_last_lr()[0]:.2e}  ({time.time()-t0:.0f}s)')

    _save(LAST_CKPT, epoch, val_loss)
    if val_loss < best_val:
        best_val = val_loss
        _save(BEST_CKPT, epoch, val_loss)
        print(f'  ✓ best checkpoint saved (val={val_loss:.4f})')

    # Sync to Drive periodically so progress survives session resets
    if drive_available and epoch % DRIVE_SYNC_EVERY == 0:
        shutil.copy(LAST_CKPT, DRIVE_LAST)
        if BEST_CKPT.exists():
            shutil.copy(BEST_CKPT, DRIVE_BEST)
        print(f'  → synced to Drive (epoch {epoch})')

print(f'Done. Best val loss: {best_val:.4f}')

In [ ]:
# ── 7. Save to Google Drive (run before closing session) ─────────────────────
# Also runs automatically every DRIVE_SYNC_EVERY epochs inside the training loop.
import shutil
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
shutil.copy('outputs/checkpoints/last.pt', '/content/drive/MyDrive/s23dr_last.pt')
shutil.copy('outputs/checkpoints/best.pt', '/content/drive/MyDrive/s23dr_best.pt')
print('Saved last.pt and best.pt to Google Drive')

In [ ]:
# ── 8. Download best checkpoint to local machine ─────────────────────────────
from google.colab import files
files.download('outputs/checkpoints/best.pt')